In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from tqdm import tqdm

# ==============================================================================
# 1. 全局参数 & H₂ 分子定义
# ==============================================================================
K = 2                      # 同时计算的态数（基态 + 第一激发态）
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# NetKet 哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

# 原始 Hilbert 空间（2个轨道，每个自旋1个电子）
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
hi_ensemble = hi ** K          # 扩展空间

# 采样器：使用 TensorRule 组合 K 个单链规则
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hilbert=hi_ensemble, rules=[single_rule]*K)
sampler = nk.sampler.MetropolisSampler(hi_ensemble, rule=tensor_rule, n_chains=16, sweep_size=32)

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV


In [2]:

# 假设已有:
# ha: 原系统的哈密顿量 (作用于 hi)
# hi: 原系统 Hilbert 空间
# K: 拷贝数
# hi_ensemble: 扩展 Hilbert 空间 (通过 hi**K 获得)

def build_ensemble_hamiltonian(ha, hi, K):
    """
    显式构造 NES-VMC 扩展哈密顿量
    H_ext = Σ_i I⊗...⊗H⊗...⊗I
    """
    I = nk.operator.Identity(hi)
    
    H_ext = None
    for i in range(K):
        # 构造第 i 个位置的项: I⊗...⊗H⊗...⊗I
        ops = [I] * K
        ops[i] = ha
        
        # 通过 @ 链式组合
        term = ops[0]
        for j in range(1, K):
            term = term @ ops[j]
        
        if H_ext is None:
            H_ext = term
        else:
            H_ext = H_ext + term
    
    return H_ext

In [3]:
I = nk.operator.Identity(hi)
term1 = ha @ I   # H ⊗ I
term2 = I @ ha   # I ⊗ H
H_ext = term1 + term2

AttributeError: module 'netket.operator' has no attribute 'Identity'

In [ ]:
import netket as nk
nk.operator._pauli_strings.PauliStrings()

In [5]:
ha@ha

TypeError: unsupported operand type(s) for @: 'ParticleNumberAndSpinConservingFermioperator2nd' and 'ParticleNumberAndSpinConservingFermioperator2nd'